# 01 — Análisis Exploratorio de Datos (EDA)
**Proyecto:** Predicción de resultados de partidos de fútbol internacional  
**Dataset:** International Football Results 1872–2024 (Kaggle)  
**Curso:** Inteligencia Artificial · EAFIT 2026-1

In [ ]:
# Instalar dependencias si estás en Google Colab
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost shap anthropic kagglehub -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'figure.dpi': 150})
sns.set_style('whitegrid')
print('Librerías cargadas ✓')

## 1. Carga del dataset

**Opción A (recomendada) — Kaggle API:**
```bash
kaggle datasets download -d martj42/international-football-results-from-1872-to-2017
```

**Opción B — Descarga directa:**  
https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017

**Opción C — Google Colab con kagglehub:**

In [ ]:
# OPCIÓN C: kagglehub (ejecutar si estás en Colab)
# import kagglehub
# path = kagglehub.dataset_download('martj42/international-football-results-from-1872-to-2017')
# import os; files = os.listdir(path); print(files)
# df_raw = pd.read_csv(f'{path}/results.csv')

# OPCIÓN LOCAL: si ya descargaste el CSV
# df_raw = pd.read_csv('../data/raw/results.csv')

# ── DEMO: generar datos sintéticos para que el notebook corra sin credenciales ──
np.random.seed(42)
n = 10000
teams = ['Brazil','Argentina','France','Germany','Spain','England','Italy',
         'Netherlands','Portugal','Uruguay','Colombia','Chile','Mexico','USA']

home = np.random.choice(teams, n)
away = np.random.choice(teams, n)
# Puntajes con ventaja local
home_score = np.random.poisson(1.6, n)
away_score = np.random.poisson(1.1, n)
dates = pd.date_range('1990-01-01', periods=n, freq='3D')

df_raw = pd.DataFrame({
    'date': dates,
    'home_team': home,
    'away_team': away,
    'home_score': home_score,
    'away_score': away_score,
    'tournament': np.random.choice(
        ['FIFA World Cup','Friendly','UEFA Euro','Copa America','Qualifier'], n,
        p=[0.05, 0.40, 0.10, 0.10, 0.35]
    ),
    'neutral': np.random.choice([True, False], n, p=[0.3, 0.7])
})

print(f'Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
df_raw.head()

## 2. Información general del dataset

In [ ]:
print('=== Tipos de columnas ===')
print(df_raw.dtypes)
print(f'\n=== Valores nulos ===')
print(df_raw.isnull().sum())
print(f'\n=== Estadísticas básicas ===')
df_raw.describe()

## 3. Variable objetivo: resultado del partido

In [ ]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

df_raw['result'] = df_raw.apply(get_result, axis=1)

# Distribución de clases
result_counts = df_raw['result'].value_counts()
print(result_counts)
print(f'\nDesbalance: {result_counts.max()/result_counts.min():.2f}x entre clase mayor y menor')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: Distribución de la variable objetivo
colors = ['#2196F3', '#FF9800', '#4CAF50']
result_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Distribución de resultados', fontweight='bold')
axes[0].set_xlabel('Resultado')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(result_counts):
    axes[0].text(i, v + 50, f'{v/len(df_raw)*100:.1f}%', ha='center', fontsize=10)

# Plot 2: Distribución de goles
axes[1].hist(df_raw['home_score'], bins=10, alpha=0.7, label='Local', color='#2196F3')
axes[1].hist(df_raw['away_score'], bins=10, alpha=0.7, label='Visitante', color='#FF9800')
axes[1].set_title('Distribución de goles', fontweight='bold')
axes[1].set_xlabel('Goles')
axes[1].legend()

# Plot 3: Resultados por tipo de torneo
tournament_result = df_raw.groupby(['tournament','result']).size().unstack(fill_value=0)
tournament_pct = tournament_result.div(tournament_result.sum(axis=1), axis=0)
tournament_pct.plot(kind='bar', ax=axes[2], color=colors, edgecolor='white')
axes[2].set_title('Resultados por torneo (%)', fontweight='bold')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=30)
axes[2].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('../docs/eda_distribucion.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada en docs/eda_distribucion.png ✓')

## 4. Hallazgos principales del EDA

In [ ]:
# Top equipos por partidos jugados
top_home = df_raw['home_team'].value_counts().head(10)
top_away = df_raw['away_team'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

top_home.plot(kind='barh', ax=axes[0], color='#003D79')
axes[0].set_title('Top 10 equipos locales', fontweight='bold')
axes[0].set_xlabel('Partidos')

# Ventaja de jugar como local
neutral_effect = df_raw.groupby('neutral')['result'].value_counts(normalize=True).unstack()
neutral_effect.index = ['Con localía', 'Campo neutro']
neutral_effect.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Efecto de la localía', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../docs/eda_localidad.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada ✓')

# Estadísticas relevantes
home_win_pct = (df_raw['result'] == 'home_win').mean() * 100
draw_pct = (df_raw['result'] == 'draw').mean() * 100
away_win_pct = (df_raw['result'] == 'away_win').mean() * 100

print(f'\n📊 HALLAZGOS CLAVE:')
print(f'  • Victoria local:     {home_win_pct:.1f}%')
print(f'  • Empate:             {draw_pct:.1f}%')
print(f'  • Victoria visitante: {away_win_pct:.1f}%')
print(f'  • Total de partidos:  {len(df_raw):,}')
print(f'  • Equipos únicos:     {df_raw["home_team"].nunique()}')

In [ ]:
# Guardar datos procesados para el siguiente notebook
df_raw.to_csv('../data/processed/football_with_result.csv', index=False)
print('Datos guardados en data/processed/football_with_result.csv ✓')